## Dataloader


In [1]:
import gc
import os
import math
import random
import warnings
import sys
from itertools import chain
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
import torch.distributed as dist

In [2]:
DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_UNK_TOKEN = "<unk>"

DEFAULT_TOKENS = {  "pad_token": DEFAULT_PAD_TOKEN,
                    "eos_token": DEFAULT_EOS_TOKEN,
                    "bos_token": DEFAULT_BOS_TOKEN,
                    "unk_token": DEFAULT_UNK_TOKEN}

In [45]:
model_name = "meta-llama/Llama-3.2-3B"
text_column = "text_column"
target_column = "target"
train_file = ""
each_train_file_percentage = 100
val_file = ""
test_file = ""
train_batch_size = 1
perplexity_eval_batch_size = 1
generative_eval_batch_size = 1
seed = 56
max_train_samples = 5000
max_eval_samples = 500
max_predict_samples = 20
# config_type = AdvanceInstructSample
task_type = "CAUSAL_LM"
block_size = 768
# no_preprocess_data = args.no_preprocess_data
# do_group_texts = args.do_group_texts
do_perplexity_eval = True
do_generative_eval = True
model_max_length = 1256
context_length = 1256
response_template = "%%%%%%% Response:"
add_tokens_list = "####### Instruction:" "%%%%%%% Response:"
max_eval_generative_samples = 10
max_eval_perplexity_samples = 300
use_fast_tokenizer = True
no_preprocess_data = False

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name,
										use_fast=use_fast_tokenizer,
										trust_remote_code=True,
										clean_up_tokenization_spaces=True,
										max_model_length=model_max_length,
										# GPT-2 is a model with absolute position embeddings so it’s
										# usually advised to pad the inputs on the right rather than the left.
										padding_side="left" if task_type == "CAUSAL_LM" and "gpt2" not in model_name else "right")

In [ ]:
tokenizer

In [30]:
for key, value in DEFAULT_TOKENS.items():
	if not getattr(tokenizer, key, None):
		print(f" {model_name}'s tokenizer does not have {key} token, setting it to {value}\n")
		setattr(tokenizer, key, value)
		tokenizer.add_special_tokens({key: value})

 meta-llama/Llama-3.2-3B's tokenizer does not have pad_token token, setting it to [PAD]

 meta-llama/Llama-3.2-3B's tokenizer does not have unk_token token, setting it to <unk>



In [32]:
if add_tokens_list:
	print(f"Adding {add_tokens_list} to the tokenizer\n")
	tokenizer.add_tokens(add_tokens_list)

Adding ####### Instruction:%%%%%%% Response: to the tokenizer



In [38]:
if torch.backends.mps.is_available():
	device = 'mps'
else:
	device = 'cpu'

print(f"Using device: {device}")

Using device: mps


In [39]:
global rank

In [43]:
if dist.is_initialized():
	rank = dist.get_rank()
else:
	rank = 0

rank

0

In [46]:
if no_preprocess_data:
	warnings.warn(f"\n Preprocessing data disable, this may result in vram accumulation overtime"
					f"Please consider enable if the size of your dataset is smaller than 1000k or your setup"
					f"have lowram\n")

In [47]:
if not max_eval_generative_samples:
	max_eval_generative_samples = max_eval_samples
else:
	assert max_eval_samples >= max_eval_generative_samples, "Max eval generative samples can't be larger than the" \
															"whole eval dataset"
	max_eval_generative_samples = max_eval_generative_samples
if not max_eval_perplexity_samples:
	max_eval_perplexity_samples = max_eval_samples
else:
	assert max_eval_samples >= max_eval_perplexity_samples, "Max eval perplexity samples can't be larger than the" \
															"whole eval dataset"
	max_eval_perplexity_samples = max_eval_perplexity_samples
max_predict_samples = max_predict_samples

In [54]:
if block_size is None:
	block_size = tokenizer.model_max_length
	if block_size > 1024:
		warnings.warn(
			"The chosen tokenizer supports a `model_max_length` that is longer than the default `block_size` value"
			" of 1024. If you would like to use a longer `block_size` up to `tokenizer.model_max_length` you can"
			" override this default with `--block_size xxx`."
		)
	block_size = 1024
else:
	if block_size > tokenizer.model_max_length:
		warnings.warn(
			f"The block_size passed ({block_size}) is larger than the maximum length for the model"
			f"({tokenizer.model_max_length}). Using block_size={tokenizer.model_max_length}."
		)
	block_size = min(block_size, tokenizer.model_max_length)
block_size

768

/var/folders/zc/5nx1mkxx4d59s0t2hw_4tzp40000gn/T/ipykernel_30489/2554615680.py:1: UserWarning: The chosen tokenizer supports a `model_max_length` that is longer than the default `block_size` value of 1024. If you would like to use a longer `block_size` up to `tokenizer.model_max_length` you can override this default with `--block_size xxx`.
  warnings.warn(


# 1

In [ ]:
import random
import sys
import warnings
from functools import partialmethod
from abc import ABC, abstractmethod
sys.path.insert(0, r'./')
from dataclasses import field

from dataclasses import dataclass



NO_ANS_RESPONSE1 = f"Tôi không thể tìm thấy câu trả lời cho câu hỏi hoặc yêu cầu '[QUESTION]' của bạn." \
                   f" Bạn có thể hỏi cụ thể hơn hoặc hỏi tôi một câu hỏi hoặc yêu cầu khác được không?"

NO_ANS_RESPONSE2 = f"Tôi không tìm thấy đáp án cho câu trả lời của bạn trong database." \
                   f" Bạn có thể đưa cho tôi thêm các tài liệu liên quan để tôi bổ sung được không?"

NO_ANS_RESPONSE3 = f"Tôi không thể trả lời câu hỏi hoặc yêu cầu của bạn, có thể do dữ liệu của tôi trong database bị thiếu hụt."    \
                   f" Bạn có thể hỏi tôi một câu hỏi hoặc yêu cầu khác được không?"

NO_ANS_RESPONSE4 = f"Câu trả lời không tìm thấy trong dữ liệu hiện có của tôi." \
                   f" Bạn có thể hỏi tôi một câu hỏi hoặc yêu cầu khác hoặc cung cấp thêm context cho tôi được không?"

NO_ANS_RESPONSE5 = f"Xin lỗi, tôi không có thông tin cần thiết để trả lời câu hỏi hoặc yêu cầu '[QUESTION]'."   \
                   f" Có thể bạn muốn thử đặt câu hỏi hoặc yêu cầu khác hoặc cung cấp thêm context để tôi hiểu rõ hơn."

NO_ANS_RESPONSE6 = f"Xin lỗi, nhưng tôi không thể cung cấp câu trả lời cho câu hỏi hoặc yêu cầu này '[QUESTION]'."  \
                   f" Bạn có thể thử đặt câu hỏi hoặc yêu cầu khác cho tôi không?"

NO_ANS_RESPONSE7 = f"Tôi rất tiếc, nhưng tôi không có thông tin cụ thể nào liên quan đến câu hỏi hoặc yêu cầu '[QUESTION]'."    \
                   f" Có câu hỏi hoặc yêu cầu khác nào tôi có thể giúp đỡ?"

NO_ANS_RESPONSE8 = f"Câu trả lời cho câu hỏi hoặc yêu cầu này '[QUESTION]' không có trong database của tôi."    \
                   f" Hãy thử hỏi điều gì đó khác nhé!"

NO_ANS_RESPONSE9 = f"Tôi không thể tìm thấy thông tin liên quan đến câu hỏi hoặc yêu cầu '[QUESTION]' trong nguồn dữ liệu hiện có." \
                   f" Có thể bạn muốn thử hỏi câu hỏi hoặc yêu cầu khác?"

NO_ANS_RESPONSE10 = f"Rất tiếc, tôi không có đủ thông tin để trả lời câu hỏi hoặc yêu cầu của bạn." \
f" Bạn có thể đưa thêm chi tiết hoặc hỏi một câu hỏi hoặc yêu cầu khác."

NO_ANS_RESPONSE11 = f"Tôi không thể cung cấp câu trả lời chính xác cho câu hỏi hoặc yêu cầu này '[QUESTION]'."  \
f" Bạn có thể cung cấp thêm thông tin để tôi hiểu rõ hơn không?"

NO_ANS_RESPONSE12 = f"Xin lỗi, tôi không có thông tin cần thiết để đáp ứng yêu cầu của bạn."    \
f" Bạn có thể thử hỏi câu hỏi hoặc yêu cầu khác hay không?"

NO_ANS_RESPONSE13 = f"Tôi không thể tìm thấy câu trả lời trong database của mình."  \
f" Hãy thử đưa ra câu hỏi hoặc yêu cầu khác để tôi có thể giúp bạn."

NO_ANS_RESPONSE14 = f"Tôi không thể cung cấp câu trả lời đầy đủ cho câu hỏi hoặc yêu cầu của bạn."  \
f" Bạn có thể cung cấp thêm thông tin hoặc hỏi câu hỏi hoặc yêu cầu khác."

NO_ANS_RESPONSE15 = f"Xin lỗi, nhưng tôi không thể tìm thấy câu trả lời."   \
f" Bạn có thể đưa thêm ngữ cảnh hoặc hỏi một câu hỏi hoặc yêu cầu khác được không?"

NO_ANS_RESPONSE16 = f"Rất tiếc, tôi không thể tìm thấy câu trả lời mà bạn đang tìm kiếm."   \
f" Có câu hỏi hoặc yêu cầu khác nào tôi có thể giúp đỡ?"

NO_ANS_RESPONSE17 = f"Tôi không có thông tin cụ thể về câu hỏi hoặc yêu cầu này '[QUESTION]'."  \
f" Bạn có thể đưa thêm thông tin chi tiết hoặc hỏi một câu hỏi hoặc yêu cầu khác."

NO_ANS_RESPONSE18 = f"Tôi không thể trả lời câu hỏi hoặc yêu cầu của bạn dựa trên dữ liệu hiện có." \
f" Hãy thử đưa thêm ngữ cảnh hoặc hỏi một câu hỏi hoặc yêu cầu khác."

NO_ANS_RESPONSE19 = f"Xin lỗi, tôi không có đủ thông tin để cung cấp câu trả lời cho câu hỏi hoặc yêu cầu này '[QUESTION]'."    \
f" Bạn có thể hỏi một câu hỏi hoặc yêu cầu khác không?"

NO_ANS_RESPONSE20 = f"Tôi không thể tìm thấy câu trả lời cho câu hỏi hoặc yêu cầu của bạn." \
f" Bạn có thể cung cấp thêm thông tin hoặc hỏi một câu hỏi hoặc yêu cầu khác."


TRIVIAL_ANS1 = f"Tôi không thể tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], tuy nhiên, "   \
f"theo tôi được biết [ANSWER]."

TRIVIAL_ANS2 = f"Kiến thức trong database không chứa thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], " \
f"nhưng tôi có thể trả lời câu hỏi này [ANSWER]."

TRIVIAL_ANS3 = f"Tôi không thể tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. "  \
f"Theo tôi được biết [ANSWER]."

TRIVIAL_ANS4 = f"Tôi không tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. [ANSWER]."

TRIVIAL_ANS5 = f"Trong database của tôi không có thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], " \
f"nhưng câu trả lời của tôi là [ANSWER]."

TRIVIAL_ANS6 = f"Tôi không tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], tuy nhiên, "   \
f"theo kiến thức của tôi, [ANSWER]."

TRIVIAL_ANS7 = f"Có vẻ như tôi không thể truy xuất được thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], "  \
f"nhưng đáp án có thể là [ANSWER]."

TRIVIAL_ANS8 = f"Xin lỗi, tôi không thể tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. " \
f"Tuy nhiên, đáp án có thể là [ANSWER]."

TRIVIAL_ANS9 = f"Tôi không thể tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], "  \
f"nhưng theo kiến thức hiện có của tôi, câu trả lời là [ANSWER]."

TRIVIAL_ANS10 = f"Không có thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], "   \
f"nhưng tôi cho rằng câu trả lời là [ANSWER]."

TRIVIAL_ANS11 = f"Tôi không tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION], " \
f"nhưng theo kiến thức của tôi, câu trả lời có thể là [ANSWER]."

TRIVIAL_ANS12 = f"Rất tiếc, không có thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. " \
f"Tuy nhiên, theo tôi được biết, câu trả lời là [ANSWER]."

TRIVIAL_ANS13 = f"Không tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. " \
f"Tuy vậy, tôi nghĩ câu trả lời là [ANSWER]."

TRIVIAL_ANS14 = f"Xin lỗi, tôi không thể tìm thấy thông tin về câu hỏi hoặc yêu cầu [QUESTION], "   \
f"nhưng dựa vào kiến thức hiện có, câu trả lời là [ANSWER]."

TRIVIAL_ANS15 = f"Trong database của tôi không có thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. "    \
f"Nhưng theo tôi được biết, câu trả lời là [ANSWER]."

TRIVIAL_ANS16 = f"Xin lỗi, tôi không thể tìm thấy thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. "    \
f"Tuy nhiên, tôi nghĩ câu trả lời có thể là [ANSWER]."

TRIVIAL_ANS17 = f"Không có thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION] trong database của tôi. "    \
f"Tuy vậy, tôi nghĩ câu trả lời là [ANSWER]."

TRIVIAL_ANS18 = f"Tôi đã kiểm tra nhưng không có thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. " \
f"Theo tôi được biết, câu trả lời là [ANSWER]."

TRIVIAL_ANS19 = f"Tôi không tìm thấy thông tin về câu hỏi hoặc yêu cầu [QUESTION]. "    \
f"Nhưng dựa vào kiến thức hiện có, câu trả lời là [ANSWER]."

TRIVIAL_ANS20 = f"Rất tiếc, không có thông tin cụ thể về câu hỏi hoặc yêu cầu [QUESTION]. " \
f"Tuy nhiên, tôi cho rằng câu trả lời là [ANSWER]."


RESPONSE1 = f"Dựa vào thông tin hiện có, tôi nghĩ câu trả lời có thể là: [ANSWER]"

RESPONSE2 = f"[ANSWER]"

RESPONSE3 = f"Theo tôi, có thể câu trả lời là: [ANSWER]"

RESPONSE4 = f"Điều mà tôi có thể kết luận là: [ANSWER]"

RESPONSE5 = f"[ANSWER]"

RESPONSE6 = f"Có nhiều khả năng câu trả lời là: [ANSWER]"

RESPONSE7 = f"Dựa vào dữ kiện, tôi suy luận câu trả lời là: [ANSWER]"

RESPONSE8 = f"Với những thông tin hiện có, tôi đánh giá là: [ANSWER]"

RESPONSE9 = f"Tôi có cảm giác câu trả lời có thể là: [ANSWER]"

RESPONSE10 = f"Dựa vào kiến thức, tôi đưa ra dự đoán là: [ANSWER]"

RESPONSE11 = f"Theo những gì tôi biết, có thể câu trả lời là: [ANSWER]"

RESPONSE12 = f"Từ dữ liệu có sẵn, tôi cho rằng câu trả lời là: [ANSWER]"

RESPONSE13 = f"[ANSWER]"

RESPONSE14 = f"Nhìn vào dữ liệu, tôi đánh giá là: [ANSWER]"

RESPONSE15 = f"[ANSWER]"

RESPONSE16 = f"Không chắc chắn, nhưng dựa vào thông tin hiện có, câu trả lời có thể là: [ANSWER]"

RESPONSE17 = f"Theo tôi, câu trả lời có thể nằm ở: [ANSWER]"

RESPONSE18 = f"Dựa vào dữ liệu có sẵn, tôi đánh giá rằng câu trả lời là: [ANSWER]"

RESPONSE19 = f"Xét về khả năng, tôi cho rằng câu trả lời là: [ANSWER]"

RESPONSE20 = f"Dựa vào thông tin hiện tại, tôi suy đoán câu trả lời là: [ANSWER]"


PROMPT_INPUT1 = f"Dựa vào context sau: [CONTEXT] bạn hãy trả lời hoặc thực hiện yêu cầu sau: [QUESTION]," \
                f" nếu không có câu trả lời bạn có thể trả lời dựa trên kiến thức của bạn" \
                f" hoặc trả lời không tìm thấy ví dụ như: 'không tìm thấy cảu trả lời cho câu hỏi của bạn' [EOS]"

PROMPT_INPUT2 = f"Dựa trên kiến thức lấy được từ database: [CONTEXT], hãy trích xuất thông tin ra và trả lời câu hỏi " \
                f"hoặc thực hiện yêu cầu sau [QUESTION], nếu như không tìm thấy câu trả lời, bạn có thể trả lời 'không biết,..." \
                f" hoặc cố gắng trả lời với kiến thức của bạn [EOS]"

PROMPT_INPUT3 = f"Người dùng hỏi câu hỏi hoặc đưa ra yêu cầu sau: [QUESTION]," \
                f" dựa trên kiến thức lấy được từ database: [CONTEXT], bạn hãy trả lời câu hỏi đó hoặc thực hiện yêu cầu." \
                f" Nếu như không trả lời được, bạn có thể yêu cầu thêm dữ liệu hoặc cố trả lời với kiến thức của bạn [EOS]"

PROMPT_INPUT4 = f"Bây giờ, hãy tập trung vào câu hỏi hoặc yêu cầu sau: [QUESTION]. Context cho câu hỏi này là [CONTEXT]. " \
                f"Nếu không tìm thấy câu trả lời, bạn có thể nói 'Câu trả lời không tìm thấy trong dữ liệu hiện " \
                f"có của tôi, bạn có thể hỏi tôi một câu hỏi khác hoặc cung cấp thêm context cho tôi được không?' " \
                f"và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi một câu hỏi khác. Hoặc cố trả lời " \
                f"với kiến thức của bạn nếu như câu hỏi đó dễ [EOS]"

PROMPT_INPUT5 = f"Hãy xem xét câu hỏi hoặc yêu cầu này: [QUESTION]. Bạn có thể tìm câu trả lời trong dữ liệu này: [CONTEXT]. " \
                f"Nếu không có câu trả lời trong dữ liệu, bạn có thể nói 'Xin lỗi, tôi không có thông tin cần thiết để trả lời " \
                f"câu hỏi hoặc thực hiện yêu cầu của bạn. Có thể bạn muốn thử đặt câu hỏi khác hoặc cung cấp thêm context để tôi hiểu " \
                f"rõ hơn.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT6 = f"Xin hãy giúp trả lời câu hỏi hoặc thực hiện yêu cầu sau: [QUESTION]." \
                f" Dựa vào dữ kiện sau đây: [CONTEXT], bạn có thể trả lời câu hỏi hoặc thực hiện yêu cầu này." \
                f" Nếu không có câu trả lời, bạn có thể nói 'Xin lỗi, nhưng tôi không thể cung cấp câu trả lời" \
                f" cho câu hỏi hoặc thực hiện yêu cầu này.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi " \
                f"một câu hỏi khác. [EOS]"

PROMPT_INPUT7 = f"Giúp tôi trả lời câu hỏi hoặc thực hiện yêu cầu sau: [QUESTION], dựa vào dữ liệu lấy từ: [CONTEXT]. " \
                f"Nếu không biết, bạn có thể nói 'Tôi rất tiếc, nhưng tôi không có thông tin cụ thể nào liên quan" \
                f" đến câu hỏi hoặc yêu cầu của bạn.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc" \
                f" hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT8 = f"Dựa vào thông tin sau: [CONTEXT], bạn hãy trả lời câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "   \
                f"Nếu không có câu trả lời trong dữ liệu, bạn có thể cố gắng trả lời với kiến thức của bạn, hoặc "  \
                f"bạn có thể nói 'Câu trả lời cho câu hỏi này không có trong database của tôi.' và yêu cầu người "  \
                f"dùng cung cấp thông tin bổ sung hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT9 = f"Dựa vào thông tin từ database: [CONTEXT], hãy trả lời câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "   \
                f"Nếu không trả lời được, bạn có thể nói 'Tôi không thể tìm thấy thông tin liên quan đến câu hỏi "  \
                f"của bạn trong nguồn dữ liệu hiện có.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc " \
                f"hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT10 = f"Đối với câu hỏi sau hoặc yêu cầu sau: [QUESTION], bạn có thể tìm câu trả lời trong dữ liệu này: [CONTEXT]. " \
                 f"Nếu không biết, bạn có thể nói 'Rất tiếc, tôi không có đủ thông tin để trả lời câu hỏi hoặc " \
                 f"thực hiện yêu cầu của bạn.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT11 = f"Bạn hãy trả lời câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION], dựa vào kiến thức lấy được từ database: [CONTEXT]. " \
                 f"Nếu không trả lời được, bạn có thể nói 'Tôi không thể cung cấp câu trả lời chính xác cho câu "    \
                 f"hỏi hoặc yêu cầu này.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT12 = f"Hãy giúp trả lời câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION], dựa vào thông tin sau: [CONTEXT]. " \
                 f"Nếu không có câu trả lời, bạn có thể nói 'Xin lỗi, tôi không có thông tin cần thiết để đáp ứng "  \
                 f"yêu cầu của bạn.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT13 = f"Trả lời câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION], dựa vào kiến thức từ database: [CONTEXT]. "  \
                 f"Nếu không trả lời được, bạn có thể nói 'Tôi không thể tìm thấy câu trả lời trong database của mình.' "    \
                 f"và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT14 = f"Dựa vào thông tin sau: [CONTEXT], bạn có thể cung cấp câu trả lời đầy đủ cho câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION] "    \
                 f"không? Nếu không có câu trả lời, bạn có thể nói 'Tôi không thể cung cấp câu trả lời đầy đủ cho câu "  \
                 f"hỏi hoặc yêu cầu của bạn.' và yêu cầu người dùng cung cấp thông tin bổ sung hoặc hỏi câu hỏi khác. [EOS]"

PROMPT_INPUT15 = f"Xem xét thông tin sau: [CONTEXT]. "  \
                 f"Hãy cố gắng tìm kiếm câu trả lời cho câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "    \
                 f"Nếu bạn cần thêm thông tin, đừng ngần ngại yêu cầu hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT16 = f"Xem xét thông tin sau: [CONTEXT]. "  \
                 f"Tìm kiếm cẩn thận để giúp tôi trả lời câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "   \
                 f"Nếu không tìm thấy, đừng lo lắng, cứ hỏi thêm hoặc đưa ra câu hỏi khác. [EOS]"

PROMPT_INPUT17 = f"Xem xét thông tin sau: [CONTEXT]. "  \
                 f"bạn hãy cố gắng giúp tôi tìm câu trả lời cho câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "    \
                 f"Nếu thông tin không đủ, hãy cung cấp thêm chi tiết hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT18 = f"Xem xét thông tin sau: [CONTEXT]. "  \
                 f"Tôi sẽ cùng bạn tìm kiếm câu trả lời cho câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "    \
                 f"Nếu cần thông tin bổ sung, hãy yêu cầu hoặc hỏi một câu hỏi khác. [EOS]"

PROMPT_INPUT19 = f"Xem xét thông tin sau: [CONTEXT]. "  \
                 f"Hãy giúp tôi tìm câu trả lời cho câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "    \
                 f"Nếu không tìm thấy, bạn có thể bảo người dùng hỏi một câu hỏi khác [EOS]"

PROMPT_INPUT20 = f"Xem xét thông tin sau: [CONTEXT]. "  \
                 f"Hãy cùng tôi tìm câu trả lời cho câu hỏi sau hoặc thực hiện yêu cầu sau: [QUESTION]. "    \
                 f"Nếu cần thông tin bổ sung, hãy yêu cầu hoặc hỏi một câu hỏi khác. [EOS]"

GENERIC_SYSTEM_PROMPT1 = "You are an AI assistant. You will be given a task. You must generate an answer."
GENERIC_SYSTEM_PROMPT2 = "As an AI assistant, your objective is to provide a response to the given task."
GENERIC_SYSTEM_PROMPT3 = "Your role is that of an AI assistant. Your task is to produce a coherent answer to the following."
GENERIC_SYSTEM_PROMPT4 = "In this scenario, you are functioning as an AI assistant. Your duty is to generate a response."
GENERIC_SYSTEM_PROMPT5 = "Imagine you are an AI assistant tasked with generating a reply. Provide a response."
GENERIC_SYSTEM_PROMPT6 = "Your function is that of an AI assistant. Your mission is to generate a response."
GENERIC_SYSTEM_PROMPT7 = "In your role as an AI assistant, you are responsible for providing a well-formed response."
GENERIC_SYSTEM_PROMPT8 = "As an AI assistant, your goal is to generate an informative response to the following task."
GENERIC_SYSTEM_PROMPT9 = "You are functioning as an AI assistant. Your objective is to produce a relevant answer."
GENERIC_SYSTEM_PROMPT10 = "Imagine you are an AI assistant. Your task is to generate a response to the following."
GENERIC_SYSTEM_PROMPT11 = "Your role is that of an AI assistant. Your duty is to provide a response."
GENERIC_SYSTEM_PROMPT12 = "In this scenario, you are functioning as an AI assistant. Your mission is to generate a reply."
GENERIC_SYSTEM_PROMPT13 = "Your function is that of an AI assistant. Your responsibility is to generate a coherent answer."
GENERIC_SYSTEM_PROMPT14 = "In your role as an AI assistant, you are tasked with providing a well-structured response."
GENERIC_SYSTEM_PROMPT15 = "As an AI assistant, your goal is to generate a meaningful response to the following task."
GENERIC_SYSTEM_PROMPT16 = "You are functioning as an AI assistant. Your task is to produce a relevant answer."
GENERIC_SYSTEM_PROMPT17 = "Imagine you are an AI assistant. Your objective is to generate a response."
GENERIC_SYSTEM_PROMPT18 = "Your role is that of an AI assistant. Your responsibility is to provide a response."
GENERIC_SYSTEM_PROMPT19 = "In this scenario, you are functioning as an AI assistant. Your goal is to generate a reply."
GENERIC_SYSTEM_PROMPT20 = "Your function is that of an AI assistant. Your task is to provide a well-informed answer."

NO_DOCS_MESSAGE1 = f" Không documents nào có điểm đủ cao để query cho câu hỏi. "
NO_DOCS_MESSAGE2 = f" Database không chứa documents nào phù hợp cho câu hỏi. "


@dataclass
class TEMPLATE(ABC):
    all_attr: dict = field(default_factory=lambda: dict(globals().items()))
    max_template: int = 20

    def __post_init__(self):
        pass

    @abstractmethod
    def get(self, id: int, type: str=None, **kwargs):
        assert id <= self.max_template, "Invalid template id"
        assert type is not None, "Please specified the type of template"
        existed = False
        for existed_type in dict(self.all_attr).keys():
            if type in existed_type:
                existed = True
        assert existed, "The template type provided does not exist"
        pass

    @classmethod
    @property
    def get_random_id(cls) -> int:
        return random.randint(1, cls.max_template)


class QA_TEMPLATE(TEMPLATE):
    def get(self, id: int, type: str, question: str=None,
            context: str=None, answer: str=None):
        super(QA_TEMPLATE, self).get(id=id, type=type)
        # template = dict(super(QA_TEMPLATE, self).all_attr)[type+str(id)]
        template = dict(self.all_attr)[type+str(id)]

        if question:
            template = template.replace("[QUESTION]", question)
        if context:
            template = template.replace("[CONTEXT]", context)
        if answer:
            template = template.replace("[ANSWER]", answer)

        if "[QUESTION]" in template or "[CONTEXT]" in template or "[ANSWER]" in template:
            warnings.warn("Missing field(s) in template!")
        return template

    get_generic_system_prompt = partialmethod(get, answer=None,
                                                   context=None,
                                                   question=None,
                                                   type="GENERIC_SYSTEM_PROMPT")
    get_prompt = partialmethod(get, answer=None, type="PROMPT_INPUT")
    get_neg_response = partialmethod(get, answer=None, context=None, type="NO_ANS_RESPONSE")
    get_trivial_response = partialmethod(get, context=None, type="TRIVIAL_ANS")
    get_norm_response = partialmethod(get, question=None, context=None, type="RESPONSE")
    get_no_docs_msg = partialmethod(get, question=None, context=None, answer=None, type="NO_DOCS_MESSAGE")

    get_random_generic_system_prompt = partialmethod(get_generic_system_prompt, id=TEMPLATE.get_random_id)
    get_random_prompt = partialmethod(get_prompt, id=TEMPLATE.get_random_id)
    get_random_neg_response = partialmethod(get_neg_response, id=TEMPLATE.get_random_id)
    get_random_trivial_response = partialmethod(get_trivial_response, id=TEMPLATE.get_random_id)
    get_random_norm_response = partialmethod(get_norm_response, id=TEMPLATE.get_random_id)


if __name__ == "__main__":
    python_questions = [
        "How do you comment a single line in Python?",
        "Explain the difference between Python 2 and Python 3.",
        "How can you generate a random number in Python?",
        "What is the purpose of the if __name__ == '__main__': statement in Python?",
        "How do you open and read a file in Python?",
        "Explain the usage of the range() function in Python.",
        "What are Python decorators, and how are they used?",
        "How do you handle exceptions in Python?",
        "Explain the differences between lists and tuples in Python.",
        "What is a virtual environment in Python, and why is it useful?",
        "How can you remove duplicates from a list in Python?",
        "Describe the differences between append(), extend(), and insert() in Python lists.",
        "How do you iterate over a dictionary in Python?",
        "Explain the concept of a lambda function in Python.",
        "How can you find the length of a string in Python?",
        "What is the purpose of the pass statement in Python?",
        "How do you create a class and define methods in Python?",
        "Explain the difference between shallow and deep copy of objects in Python.",
        "How do you sort a dictionary by its values in Python?",
        "What is the use of the zip() function in Python?"
    ]
    python_question_contexts = [
        "To comment a single line in Python, you use the '#' symbol. Anything after the '#' on the same line is considered a comment and is ignored by the Python interpreter.",
        "Python 2 and Python 3 are two different versions of the Python programming language. Python 3 introduced several backward-incompatible changes to the language to improve its design and fix inconsistencies.",
        "To generate a random number in Python, you can use the 'random' module. Import the module and then use functions like 'random.random()' or 'random.randint()' depending on your requirements.",
        "The 'if __name__ == '__main__':' statement is used to determine whether the Python script is being run as the main program or if it is being imported as a module into another program.",
        "To open and read a file in Python, you can use the 'open()' built-in function in combination with various file modes. After reading, close the file using the 'close()' method of the file object.",
        "The 'range()' function generates a sequence of numbers in Python. It can be used in 'for' loops or to create lists of numbers within a specified range.",
        "Decorators in Python are a powerful way to modify or extend the behavior of functions or methods without changing their actual code. They use the '@' symbol and can be used to add functionality like logging, caching, etc.",
        "Exception handling in Python is done using 'try', 'except', 'else', and 'finally' blocks. It allows you to gracefully handle errors and exceptions that may occur during program execution.",
        "Lists and tuples are both used to store collections of items in Python, but lists are mutable (can be changed), whereas tuples are immutable (cannot be changed).",
        "A virtual environment in Python is a self-contained directory that contains its own Python interpreter and installed packages. It allows you to work on different projects with different dependencies without conflicts.",
        "To remove duplicates from a list in Python, you can convert it to a 'set' and then back to a 'list', as sets automatically remove duplicate elements.",
        "In Python lists, 'append()' adds an element to the end of the list, 'extend()' appends elements from another iterable, and 'insert()' inserts an element at a specified index.",
        "To iterate over a dictionary in Python, you can use a 'for' loop, which by default, iterates over the keys. You can also use the 'items()' method to loop over both keys and values.",
        "A lambda function in Python is a small anonymous function defined using the 'lambda' keyword. It can have any number of arguments but can only have one expression.",
        "You can find the length of a string in Python using the built-in 'len()' function, which returns the number of characters in the string.",
        "The 'pass' statement in Python is a null operation; it does nothing. It is used as a placeholder where syntactically some code is required, but you want to skip its execution.",
        "In Python, you create a class using the 'class' keyword, and you define methods (functions within the class) to perform actions or return information about the object.",
        "Shallow copy creates a new object, but it references the same elements as the original object. Deep copy creates a completely independent copy of the object and its elements.",
        "To sort a dictionary by its values in Python, you can use the 'sorted()' function with a lambda function as the 'key' argument or utilize the 'collections' module.",
        "The 'zip()' function in Python is used to combine multiple iterables into a single iterable of tuples. Each tuple contains elements from corresponding positions in the input iterables."
    ]
    # TEMPLATE().get_prompt(10, question="Why is the sky red ?")
    # TEMPLATE(seed=45).get_random_prompt(question="Why is the sky blue ?")

    # prompts = TEMPLATE().get_random_norm_response_batch(answers=python_question_contexts)
    # for prompt in prompts:
    #     print(prompt+"\n")

    prompt = QA_TEMPLATE().get_trivial_response(id=20,
                                                question=python_questions[0],
                                                answer=python_question_contexts[0])
    print(prompt)

    # prompt = QA_TEMPLATE().get_random_prompt(question=python_questions[0],
    #                                          context=python_question_contexts[0])
    #
    # print(prompt)

    # prompt = QA_TEMPLATE().get_random_generic_system_prompt()
    # print(prompt)


Rất tiếc, không có thông tin cụ thể về câu hỏi hoặc yêu cầu How do you comment a single line in Python?. Tuy nhiên, tôi cho rằng câu trả lời là To comment a single line in Python, you use the '#' symbol. Anything after the '#' on the same line is considered a comment and is ignored by the Python interpreter..


In [2]:
import json
import sys
import random
sys.path.insert(0,r'./')
import pprint
from pprint import PrettyPrinter
from typing import List, Dict
from dataclasses import dataclass, field, asdict, fields


@dataclass
class AdvanceQAExample:
    """
    A single training/test example for the QA dataset.
    """
    qas_id: str
    question_text: str

    is_impossible: bool = None
    is_trivial:bool = None

    doc_tokens: List[str] = field(default_factory=list)
    docs_lengths: List[int] = None

    orig_answer_texts: str = None
    answer_lengths: int = None

    # example_template = QA_TEMPLATE()

    def __post_init__(self) -> None:
        # Post validate
        self.is_impossible = True if self.orig_answer_texts is None else False
        self.is_trivial = False if self.orig_answer_texts is None else self.is_trivial

        self.answer_lengths = len(self.orig_answer_texts) if self.orig_answer_texts is not None else None

        if self.doc_tokens:
            self.docs_lengths = [len(doc) for doc in self.doc_tokens]
            random.shuffle(self.doc_tokens)

    def __str__(self) -> str:
        return self.__repr__

    @property
    def __repr__(self) -> str:
        s = ""
        s += f"\n Question id: {self.qas_id}"
        s += f"\n Question: {self.question_text}"
        if self.doc_tokens:
            s += f"\n Doc tokens: {self.straighten_docs(self.doc_tokens)}"
            s += f"\n Doc lengths: {self.docs_lengths}"
        if self.orig_answer_texts:
            s += f"\n Answer text: {self.orig_answer_texts}"
            s += f"\n Answer length: {self.answer_lengths}"
        if self.is_impossible is not None:
            s += f"\n Is impossiple: {self.is_impossible}"
        if self.is_trivial is not None:
            s += f"\n Is trivial: {self.is_trivial} \n"

        return s

    @property
    def get_dict(self) -> Dict:
        return asdict(self)

    @staticmethod
    def get_keys() -> List[str]:
        all_fields = fields(AdvanceQAExample)
        return [v.name for v in all_fields]

    @property
    def get_dict_str(self, indent: int=4) -> None:
        pp = pprint.PrettyPrinter(indent=indent)
        pp.pprint(self.get_dict)

    def get_example(self,
                    is_training: bool=False,
                    inputs_column: str="prompt",
                    targets_column: str="target") -> Dict:
        if is_training:
            straightened_docs = self.straighten_docs(self.doc_tokens)
            prompt = QA_TEMPLATE().get_random_prompt(question=self.question_text,
                                                             context=straightened_docs)
            if not self.is_impossible:
                if self.is_trivial and not self.doc_tokens:
                    label = QA_TEMPLATE().get_random_trivial_response(question=self.question_text,
                                                                              answer=self.orig_answer_texts)
                elif self.doc_tokens:
                    label = QA_TEMPLATE().get_random_norm_response(answer=self.orig_answer_texts)
                else:
                    label = QA_TEMPLATE().get_random_neg_response(question=self.question_text)
            else:
                label = QA_TEMPLATE().get_random_neg_response(question=self.question_text)

            return {inputs_column: prompt,
                    targets_column: label}

    @staticmethod
    def straighten_docs(docs_list: List[str]) -> str:
        ctxs = []
        if not docs_list:
            return f"[ERROR]{QA_TEMPLATE().get_no_docs_msg(id=1)}[ERROR]"
        for idx, doc in enumerate(docs_list):
            ctxs.append(f" [CTX{idx}]: {doc} [ECTX{idx}] ")
        return "".join(ctxs)


if __name__ == "__main__":
    python_question_contexts = [
        "To comment a single line in Python, you use the '#' symbol. Anything after the '#' on the same line is considered a comment and is ignored by the Python interpreter.",
        "Python 2 and Python 3 are two different versions of the Python programming language. Python 3 introduced several backward-incompatible changes to the language to improve its design and fix inconsistencies.",
        "To generate a random number in Python, you can use the 'random' module. Import the module and then use functions like 'random.random()' or 'random.randint()' depending on your requirements.",
        "The 'if __name__ == '__main__':' statement is used to determine whether the Python script is being run as the main program or if it is being imported as a module into another program.",
        "To open and read a file in Python, you can use the 'open()' built-in function in combination with various file modes. After reading, close the file using the 'close()' method of the file object.",
        "The 'range()' function generates a sequence of numbers in Python. It can be used in 'for' loops or to create lists of numbers within a specified range.",
        "Decorators in Python are a powerful way to modify or extend the behavior of functions or methods without changing their actual code. They use the '@' symbol and can be used to add functionality like logging, caching, etc.",
        "Exception handling in Python is done using 'try', 'except', 'else', and 'finally' blocks. It allows you to gracefully handle errors and exceptions that may occur during program execution.",
        "Lists and tuples are both used to store collections of items in Python, but lists are mutable (can be changed), whereas tuples are immutable (cannot be changed).",
        "A virtual environment in Python is a self-contained directory that contains its own Python interpreter and installed packages. It allows you to work on different projects with different dependencies without conflicts.",
        "To remove duplicates from a list in Python, you can convert it to a 'set' and then back to a 'list', as sets automatically remove duplicate elements.",
        "In Python lists, 'append()' adds an element to the end of the list, 'extend()' appends elements from another iterable, and 'insert()' inserts an element at a specified index.",
        "To iterate over a dictionary in Python, you can use a 'for' loop, which by default, iterates over the keys. You can also use the 'items()' method to loop over both keys and values.",
        "A lambda function in Python is a small anonymous function defined using the 'lambda' keyword. It can have any number of arguments but can only have one expression.",
        "You can find the length of a string in Python using the built-in 'len()' function, which returns the number of characters in the string.",
        "The 'pass' statement in Python is a null operation; it does nothing. It is used as a placeholder where syntactically some code is required, but you want to skip its execution.",
        "In Python, you create a class using the 'class' keyword, and you define methods (functions within the class) to perform actions or return information about the object.",
        "Shallow copy creates a new object, but it references the same elements as the original object. Deep copy creates a completely independent copy of the object and its elements.",
        "To sort a dictionary by its values in Python, you can use the 'sorted()' function with a lambda function as the 'key' argument or utilize the 'collections' module.",
        "The 'zip()' function in Python is used to combine multiple iterables into a single iterable of tuples. Each tuple contains elements from corresponding positions in the input iterables."
    ]

    # print(AdvanceQAExample.straighten_docs(python_question_contexts))

    example8 = AdvanceQAExample(qas_id="8", question_text="What do cats eat?",
                                doc_tokens=[],
                                orig_answer_texts="meat and fish", is_trivial=True)

    print(example8)
    print(example8.get_example(is_training=True))
    print(example8.get_dict)

    example6 = AdvanceQAExample(qas_id="6", question_text="What is the meaning of existence?",
                                doc_tokens=["The meaning of existence is uncertain.",
                                            "Throughout history, philosophers and theologians have debated the purpose of life and the nature of existence."],
                                orig_answer_texts="Dying", is_trivial=False)
    print(example6)
    print(example6.get_example(is_training=True))
    print(example6.get_dict)
    print(example6.get_dict_str)
    print(example6.get_keys())


 Question id: 8
 Question: What do cats eat?
 Answer text: meat and fish
 Answer length: 13
 Is impossiple: False
 Is trivial: True 

{'prompt': 'Xem xét thông tin sau: [ERROR] Không documents nào có điểm đủ cao để query cho câu hỏi. [ERROR]. Tìm kiếm cẩn thận để giúp tôi trả lời câu hỏi sau hoặc thực hiện yêu cầu sau: What do cats eat?. Nếu không tìm thấy, đừng lo lắng, cứ hỏi thêm hoặc đưa ra câu hỏi khác. [EOS]', 'target': 'Rất tiếc, không có thông tin cụ thể về câu hỏi hoặc yêu cầu What do cats eat?. Tuy nhiên, tôi cho rằng câu trả lời là meat and fish.'}
{'qas_id': '8', 'question_text': 'What do cats eat?', 'is_impossible': False, 'is_trivial': True, 'doc_tokens': [], 'docs_lengths': None, 'orig_answer_texts': 'meat and fish', 'answer_lengths': 13}

 Question id: 6
 Question: What is the meaning of existence?
 Doc tokens:  [CTX0]: Throughout history, philosophers and theologians have debated the purpose of life and the nature of existence. [ECTX0]  [CTX1]: The meaning of existenc

In [3]:
import json
import sys
import random
sys.path.insert(0,r'./')
import pprint
from pprint import PrettyPrinter
from typing import List, Dict
from dataclasses import dataclass, field, asdict, fields
from peft import TaskType


@dataclass
class AdvanceInstructSample:
    """
    A single training/test example for the Instruct dataset.
    """
    qas_id: str
    system_prompt: str

    question_text: str

    orig_answer_texts: str = None
    answer_lengths: int = None

    # example_template = QA_TEMPLATE()

    def __post_init__(self) -> None:
        # Post validate
        self.answer_lengths = len(self.orig_answer_texts) if self.orig_answer_texts is not None else None

    def __str__(self) -> str:
        return self.__repr__

    @property
    def __repr__(self) -> str:
        s = ""
        s += f"\n Question id: {self.qas_id}"
        s += f"\n System prompt: {self.system_prompt}"
        s += f"\n Question: {self.question_text}"
        if self.orig_answer_texts:
            s += f"\n Answer text: {self.orig_answer_texts}"
            s += f"\n Answer length: {self.answer_lengths}"

        return s

    @property
    def get_dict(self) -> Dict:
        return asdict(self)

    @staticmethod
    def get_keys() -> List[str]:
        all_fields = fields(AdvanceInstructSample)
        return [v.name for v in all_fields]

    @property
    def get_dict_str(self, indent: int=4) -> None:
        pp = pprint.PrettyPrinter(indent=indent)
        pp.pprint(self.get_dict)

    def get_example(self,
                    inputs_column: str="prompt",
                    targets_column: str="target",
                    system_prefix: str="",
                    question_prefix: str="####### Instruction:",
                    response_prefix: str="%%%%%%% Response:",
                    is_training: bool=True,
                    do_perplexity_eval: bool=False,
                    do_generative_eval: bool=False,
                    task_type: str=None,
                    ) -> Dict:
        assert task_type, "Please specified the task type inorder to get the example"

        system_msg = ' ' + system_prefix + '\n' + self.system_prompt + "\n\n"
        question_msg = question_prefix + '\n' + self.question_text + "\n\n"
        prompt = system_msg + ' ' + question_msg
        label = self.orig_answer_texts + "\n"

        if task_type == "SEQ_2_SEQ_LM":
            return {inputs_column: prompt,
                    targets_column: label}
        elif task_type == "CAUSAL_LM":
            if is_training:
                return {inputs_column: prompt + ' ' + response_prefix + '\n' + label}

            example_dict = {}
            # The perplexity field is for perplexity evaluation, which needed the full prompt and label
            # while the inputs_column only have prompt and response_prefix for model.generate evaluation
            if do_generative_eval:
                example_dict[inputs_column] = prompt + ' ' + response_prefix + '\n'
                example_dict[targets_column] = label

            if do_perplexity_eval:
                example_dict["perplexity"] = prompt + ' ' + response_prefix + '\n' + label

            if not bool(example_dict):
                raise "Evaluation files is provided but don't know what to do with them..."

            return example_dict
        else:
            raise f"This task type {task_type} is not support"


if __name__ == "__main__":
    python_question_contexts = [
        "To comment a single line in Python, you use the '#' symbol. Anything after the '#' on the same line is considered a comment and is ignored by the Python interpreter.",
        "Python 2 and Python 3 are two different versions of the Python programming language. Python 3 introduced several backward-incompatible changes to the language to improve its design and fix inconsistencies.",
        "To generate a random number in Python, you can use the 'random' module. Import the module and then use functions like 'random.random()' or 'random.randint()' depending on your requirements.",
        "The 'if __name__ == '__main__':' statement is used to determine whether the Python script is being run as the main program or if it is being imported as a module into another program.",
        "To open and read a file in Python, you can use the 'open()' built-in function in combination with various file modes. After reading, close the file using the 'close()' method of the file object.",
        "The 'range()' function generates a sequence of numbers in Python. It can be used in 'for' loops or to create lists of numbers within a specified range.",
        "Decorators in Python are a powerful way to modify or extend the behavior of functions or methods without changing their actual code. They use the '@' symbol and can be used to add functionality like logging, caching, etc.",
        "Exception handling in Python is done using 'try', 'except', 'else', and 'finally' blocks. It allows you to gracefully handle errors and exceptions that may occur during program execution.",
        "Lists and tuples are both used to store collections of items in Python, but lists are mutable (can be changed), whereas tuples are immutable (cannot be changed).",
        "A virtual environment in Python is a self-contained directory that contains its own Python interpreter and installed packages. It allows you to work on different projects with different dependencies without conflicts.",
        "To remove duplicates from a list in Python, you can convert it to a 'set' and then back to a 'list', as sets automatically remove duplicate elements.",
        "In Python lists, 'append()' adds an element to the end of the list, 'extend()' appends elements from another iterable, and 'insert()' inserts an element at a specified index.",
        "To iterate over a dictionary in Python, you can use a 'for' loop, which by default, iterates over the keys. You can also use the 'items()' method to loop over both keys and values.",
        "A lambda function in Python is a small anonymous function defined using the 'lambda' keyword. It can have any number of arguments but can only have one expression.",
        "You can find the length of a string in Python using the built-in 'len()' function, which returns the number of characters in the string.",
        "The 'pass' statement in Python is a null operation; it does nothing. It is used as a placeholder where syntactically some code is required, but you want to skip its execution.",
        "In Python, you create a class using the 'class' keyword, and you define methods (functions within the class) to perform actions or return information about the object.",
        "Shallow copy creates a new object, but it references the same elements as the original object. Deep copy creates a completely independent copy of the object and its elements.",
        "To sort a dictionary by its values in Python, you can use the 'sorted()' function with a lambda function as the 'key' argument or utilize the 'collections' module.",
        "The 'zip()' function in Python is used to combine multiple iterables into a single iterable of tuples. Each tuple contains elements from corresponding positions in the input iterables."
    ]

    # print(AdvanceQAExample.straighten_docs(python_question_contexts))

    example8 = AdvanceInstructSample(qas_id="8", question_text="What do cats eat?",
                                    orig_answer_texts="meat and fish", system_prompt="Hi")

    # print(example8)
    print(example8.get_example(is_training=True, task_type="CAUSAL_LM"))
    # print(example8.get_dict)

    example6 = AdvanceInstructSample(qas_id="6", question_text="What is the meaning of existence?",
                                     orig_answer_texts="Dying", system_prompt="Hello")
    # print(example6)
    print(example6.get_example(is_training=True, task_type="SEQ_2_SEQ_LM"))
    # print(example6.get_dict)
    # print(example6.get_dict_str)
    # print(example6.get_keys())

{'prompt': ' \nHi\n\n ####### Instruction:\nWhat do cats eat?\n\n %%%%%%% Response:\nmeat and fish\n'}
{'prompt': ' \nHello\n\n ####### Instruction:\nWhat is the meaning of existence?\n\n', 'target': 'Dying\n'}


In [4]:
def dist_print(string: str):
    if dist.is_initialized():
        # Running in a distributed setting (multiple processes)
        if dist.get_rank() == 0:
            # Main process (rank 0)
            print(string)
    else:
        # Running in a single-process setting
        print(string)


def in_notebook():
    """
    Returns ``True`` if the module is running in IPython kernel,
    ``False`` if in IPython shell or other Python shell.
    """
    return 'ipykernel' in sys.modules

In [8]:
import gc
import os
import math
import random
import warnings
import sys
from itertools import chain

sys.path.insert(0, r'./')

from tqdm.contrib import tzip
from typing import Optional, List, Union, Set

import numpy as np

import torch
import torch.distributed as dist
from torch.utils.data import RandomSampler, SequentialSampler
from torch.utils.data.dataloader import DataLoader, Dataset

from datasets import load_dataset
from datasets import Dataset as hfDataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, DataCollatorForLanguageModeling
from trl import DataCollatorForCompletionOnlyLM

# from src.data.configs import AdvanceQAExample, AdvanceInstructSample
# from src.utils import dist_print, in_notebook


if in_notebook():
    try:
        from tqdm.notebook import tqdm
    except ImportError as e:
        from tqdm.auto import tqdm
else:
    from tqdm.auto import tqdm


class AdvanceQa(Dataset):

    def __init__(self, json_file_paths: List[str], task_type: str,
                 config_type: Union[AdvanceQAExample, AdvanceInstructSample] = AdvanceQAExample,
                 get_example: bool = True, split: str='train', num_examples: int = 100000,
                 do_perplexity_eval: bool = False, do_generative_eval: bool = False,
                 tokenizer: AutoTokenizer = None, max_seq_length: int = 1024,
                 percentage_weights: List[int]=None):
        assert task_type, "Please specified task type"

        # Uniform weights for all files if percentage weights is None
        if not percentage_weights:
            percentage_weights = [math.floor(100/len(json_file_paths)) for _ in range(len(json_file_paths))]

        self.task_type = task_type
        self.full_json_data = []
        self.config_type = config_type
        self.get_example = get_example
        for json_path, percentage_weight in tzip(json_file_paths,
                                                 percentage_weights,
                                                 desc=f"Loading {split} data",
                                                 disable=rank!=0):
            assert os.path.isfile(json_path), f"Invalid data path for {json_path}"
            num_examples_each_file = math.floor(num_examples * (percentage_weight/100))
            loading_bar_desc = f"Loading data from {os.path.basename(json_path)} for split {split}"
            loading_bar = tqdm(total=num_examples_each_file,
                               colour="green",
                               desc=loading_bar_desc,
                               disable=rank!=0)
            total_skipped = 0
            try:
                file_name = os.path.basename(json_path)
                extension = json_path.split(".")[-1]
                dist_print(f"Loading {num_examples_each_file} examples with percentage of {percentage_weight} from {file_name}...")
                iterable_json_data = load_dataset(extension, data_files=json_path,
                                                  streaming=True, keep_in_memory=False)
                for idx, data in enumerate(iter(iterable_json_data['train'])):
                    if idx > num_examples_each_file:
                        break
                    if get_example:
                        try:
                            config_data = self.config_type(**data).get_example(is_training=split == 'train',
                                                                               task_type=self.task_type,
                                                                               do_perplexity_eval=do_perplexity_eval,
                                                                               do_generative_eval=do_generative_eval)
                            # Check if config data exceeds maximum length or not,
                            # This check is for DataCollatorForCompletionOnlyLM
                            if task_type == 'CAUSAL_LM':
                                if split == 'train' or do_generative_eval:
                                    config_data_tokenzied = tokenizer(config_data['prompt'])
                                    if len(config_data_tokenzied['input_ids']) > max_seq_length:
                                        total_skipped += 1
                                        loading_bar.desc = f"{loading_bar_desc} (Total skipped {total_skipped})"
                                        num_examples_each_file += 1
                                        del config_data_tokenzied
                                        continue
                                if do_perplexity_eval:
                                    config_data_tokenzied = tokenizer(config_data['perplexity'])
                                    if len(config_data_tokenzied['input_ids']) > max_seq_length:
                                        total_skipped += 1
                                        loading_bar.desc = f"{loading_bar_desc} (Total skipped {total_skipped})"
                                        num_examples_each_file += 1
                                        del config_data_tokenzied
                                        continue
                        except KeyError as e:
                            raise f"Missing keys to fill for {config_data} in item {idx} in {file_name}" \
                                  f"Error message: {e}"
                    else:
                        config_data = data
                    self.full_json_data.append(config_data)
                    loading_bar.update(1)
                loading_bar.close()
                dist_print(f"\nFinished loading from {file_name} with total loaded {len(self.full_json_data)} examples\n"
                           f"\nTotal data skipped: {total_skipped}\n")
                del iterable_json_data,
                gc.collect()
            except IOError as e:
                raise f"An error occurred while reading the data: {e}"

    def __len__(self) -> int:
        return len(self.full_json_data)

    def __getitem__(self, idx):
        if not self.get_example:
            try:
                config_data = self.config_type(**self.full_json_data[idx])
            except KeyError:
                raise f"Missing keys to fill for {config_data} in item {idx}"
            return config_data
        return self.full_json_data[idx]


DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_UNK_TOKEN = "<unk>"

DEFAULT_TOKENS = {  "pad_token": DEFAULT_PAD_TOKEN,
                    "eos_token": DEFAULT_EOS_TOKEN,
                    "bos_token": DEFAULT_BOS_TOKEN,
                    "unk_token": DEFAULT_UNK_TOKEN}


class  QADataloader:
    def __init__(self,
                 model_name: str,
                 text_column: str,
                 target_column: str,
                 task_type: str,
                 train_file: Union[str, List[str]],
                 each_train_file_percentage: List[int]=None,
                 val_file: Optional[Union[str, List[str]]]=None,
                 test_file: Optional[Union[str, List[str]]]=None,
                 train_batch_size: int = 8,
                 generative_eval_batch_size: int=16,
                 max_eval_generative_samples: Optional[int] = None,
                 max_eval_perplexity_samples: Optional[int] = None,
                 perplexity_eval_batch_size: int=6,
                 test_batch_size: int=16,
                 block_size: int=768,
                 model_max_length: int=1024,
                 context_length: int=768,
                 num_worker: int = 1,
                 seed: int = 42,
                 use_fast_tokenizer: bool=True,
                 no_preprocess_data: bool=False,
                 do_perplexity_eval: bool=False,
                 do_generative_eval: bool=False,
                 do_group_texts: bool=False,
                 response_template: str=" %%%%%%% Response:",
                 add_tokens_list: List[str]=None,
                 max_train_samples: Optional[int] = None,
                 max_eval_samples: Optional[int] = None,
                 max_predict_samples: Optional[int] = None,
                 config_type: Union[AdvanceQAExample, AdvanceInstructSample] = AdvanceQAExample
                 ) -> None:

        self.model_max_length = model_max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name,
                                                       use_fast=use_fast_tokenizer,
                                                       trust_remote_code=True,
                                                       clean_up_tokenization_spaces=True,
                                                       max_model_length=self.model_max_length,
                                                       # GPT-2 is a model with absolute position embeddings so it’s
                                                       # usually advised to pad the inputs on the right rather than the left.
                                                       padding_side="left" if task_type == "CAUSAL_LM" and "gpt2" not in model_name else "right")
            
        for key, value in DEFAULT_TOKENS.items():
            if not getattr(self.tokenizer, key, None):
                print(f" {model_name}'s tokenizer does not have {key} token, setting it to {value}\n")
                setattr(self.tokenizer, key, value)
                self.tokenizer.add_special_tokens({key: value})
        
        self.add_tokens_list = add_tokens_list
        if self.add_tokens_list:
            print(f"Adding {self.add_tokens_list} to the tokenizer\n")
            self.tokenizer.add_tokens(self.add_tokens_list)

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        global rank
        if dist.is_initialized():
            rank = dist.get_rank()
        else:
            rank = 0

        self.text_column = text_column
        self.response_template = response_template
        self.target_column = target_column
        self.config_type = config_type
        self.task_type= task_type
        self.block_size = block_size
        self.context_length = context_length
        self.no_preprocess_data = no_preprocess_data
        self.do_group_texts = do_group_texts
        self.do_perplexity_eval = do_perplexity_eval
        self.do_generative_eval = do_generative_eval
        if no_preprocess_data:
            warnings.warn(f"\n Preprocessing data disable, this may result in vram accumulation overtime"
                          f"Please consider enable if the size of your dataset is smaller than 1000k or your setup"
                          f"have lowram\n")
        self.train_file = train_file
        self.each_train_file_percentage = each_train_file_percentage
        self.val_file = val_file
        self.test_file = test_file
        self.train_batch_size = train_batch_size
        self.generative_eval_batch_size = generative_eval_batch_size
        self.perplexity_eval_batch_size = perplexity_eval_batch_size
        self.test_batch_size = test_batch_size

        self.seed = seed
        self.num_worker = num_worker
        self.generator = torch.Generator()
        self.generator.manual_seed(self.seed)

        self.max_train_samples = max_train_samples
        self.max_eval_samples = max_eval_samples
        if not max_eval_generative_samples:
            self.max_eval_generative_samples = max_eval_samples
        else:
            assert max_eval_samples >= max_eval_generative_samples, "Max eval generative samples can't be larger than the" \
                                                                   "whole eval dataset"
            self.max_eval_generative_samples = max_eval_generative_samples
        if not max_eval_perplexity_samples:
            self.max_eval_perplexity_samples = max_eval_samples
        else:
            assert max_eval_samples >= max_eval_perplexity_samples, "Max eval perplexity samples can't be larger than the" \
                                                                   "whole eval dataset"
            self.max_eval_perplexity_samples = max_eval_perplexity_samples
        self.max_predict_samples = max_predict_samples

        # TODO: Investigate block size string concatenation for efficient training
        if self.block_size is None:
            self.block_size = self.tokenizer.model_max_length
            if block_size > 1024:
                warnings.warn(
                    "The chosen tokenizer supports a `model_max_length` that is longer than the default `block_size` value"
                    " of 1024. If you would like to use a longer `block_size` up to `tokenizer.model_max_length` you can"
                    " override this default with `--block_size xxx`."
                )
            self.block_size = 1024
        else:
            if self.block_size > self.tokenizer.model_max_length:
                warnings.warn(
                    f"The block_size passed ({self.block_size}) is larger than the maximum length for the model"
                    f"({self.tokenizer.model_max_length}). Using block_size={self.tokenizer.model_max_length}."
                )
            self.block_size = min(self.block_size, self.tokenizer.model_max_length)

    def __call__(self, *args, **kwargs) -> Union[Set[DataLoader],Set]:
        dataloaders = {}
        self.dataset = {}
        if self.train_file is not None:
            dist_print('\nLoading train datasets' + '.' * 10)
            train_dataset = self.load_data(self.train_file, self.max_train_samples, split='train')
            dataloaders['train'] = self.get_dataloader(train_dataset if self.no_preprocess_data else self.preprocess_data(train_dataset, split='train'),
                                                       shuffle_flag=True,
                                                       batch_size=self.train_batch_size)
            self.dataset['train'] = train_dataset

        if self.val_file is not None:
            dataloaders['eval'] = {}
            dist_print('\nLoading validation datasets' + '.' * 10)
            eval_dataset = self.load_data(self.val_file,
                                          self.max_eval_samples,
                                          split='eval',
                                          do_perplexity_eval=self.do_perplexity_eval,
                                          do_generative_eval=self.do_generative_eval)
            if self.do_generative_eval or self.task_type == "SEQ_2_SEQ_LM":
                # eval_dataset_input = random.sample(list(eval_dataset), min(self.max_eval_generative_samples, len(eval_dataset)))
                eval_dataset_input = eval_dataset[:self.max_eval_generative_samples]
                eval_dataset_input = eval_dataset_input if self.no_preprocess_data else self.preprocess_data(eval_dataset_input)
                dataloaders['eval']['generative_eval'] = self.get_dataloader(eval_dataset_input,
                                                                             batch_size=self.generative_eval_batch_size)
            if self.do_perplexity_eval and not self.task_type == "SEQ_2_SEQ_LM":
                eval_dataset_input = eval_dataset[:self.max_eval_perplexity_samples]
                eval_dataset_input = eval_dataset_input if self.no_preprocess_data else self.preprocess_data(eval_dataset_input,
                                                                                        perplexity_eval=self.do_perplexity_eval)
                dataloaders['eval']['perplexity_eval'] = self.get_dataloader(eval_dataset_input,
                                                                             batch_size=self.perplexity_eval_batch_size)
            self.dataset['eval'] = eval_dataset

        if self.test_file is not None:
            dataloaders['test'] = {}
            dist_print('\nLoading test datasets' + '.' * 10)
            test_dataset = self.load_data(self.test_file,
                                          self.max_predict_samples,
                                          split='test',
                                          do_perplexity_eval=self.do_perplexity_eval,
                                          do_generative_eval=self.do_generative_eval)

            if self.do_generative_eval or self.task_type == "SEQ_2_SEQ_LM":
                dataloaders['test']['generative_eval'] = self.get_dataloader(test_dataset if self.no_preprocess_data else self.preprocess_data(test_dataset),
                                                          batch_size=self.test_batch_size)
            if self.do_perplexity_eval and not self.task_type == "SEQ_2_SEQ_LM":
                dataloaders['test']['perplexity_eval'] = self.get_dataloader(test_dataset if self.no_preprocess_data else self.preprocess_data(test_dataset,
                                                                                                                                       perplexity_eval=self.do_perplexity_eval),
                                                                     batch_size=self.test_batch_size)

            self.dataset['test'] = test_dataset

        gc.collect()

        return dataloaders

    def load_data(self, data_files: List[str], num_example: int=100000,
                  split: str='train', get_example: bool=True,
                  do_perplexity_eval: bool=False, do_generative_eval: bool=False) -> AdvanceQa:
        """
        Loads a dataset from a file on disk and returns it as a dictionary of Dataset objects.

        Args:
            data_file (List[str]): The path or paths to the data file(s) to load. If multiple is True, data_file
                                                should be a list of file paths. Otherwise, it should be a single file path.
            num_example (int): Max number of example in the final dataset that was loaded from each file

        Returns:
            A dataset object loaded from all data_files that was divided equally
        """
        dataset = AdvanceQa(json_file_paths=data_files,
                            percentage_weights=self.each_train_file_percentage if split=='train' else None,
                            num_examples=num_example,
                            config_type=self.config_type,
                            task_type=self.task_type,
                            split=split,
                            get_example=get_example,
                            do_perplexity_eval=do_perplexity_eval,
                            do_generative_eval= do_generative_eval,
                            tokenizer=self.tokenizer,
                            max_seq_length=self.model_max_length,
                            )

        # Log a few random samples from the training set:
        for index in random.sample(range(len(dataset)), 3):
            dist_print(f"Sample {index} of the training set: {dataset[index]}.")

        return dataset

    def preprocess_data(self, dataset, split=None, perplexity_eval: bool=False):
        tokenized_dataset = list(map(lambda data: self.tokenize_function(data, split, perplexity_eval), dataset))

        if self.task_type == "CAUSAL_LM" and self.do_group_texts:
            return tokenized_dataset.map(self.group_texts,
                                         desc=f"Grouping texts in chunks of {self.block_size}",
                                         batched=False,
                                         num_proc=1)

        return tokenized_dataset

    def dynamic_collate(self, batch):
        """
        A collate function that tokenizes the inputs and targets, and applies dynamic padding and truncation
        based on the maximum length in the batch.

        Args:
            batch (list): A list of examples, where each example is a dictionary with a text column and a target column.

        Returns:
            dict: A dictionary with the input IDs, attention masks, and target IDs with attention masks where tokens are
            padded and the target IDs are masked to exclude padded values.
        """

        inputs = [example[self.text_column] for example in batch]

        inp_tokens = self.tokenizer.batch_encode_plus(
            inputs,
            padding=True,
            return_tensors="pt",
            truncation=True,
        )
        if self.task_type == "SEQ_2_SEQ_LM":
            targets = [example[self.target_column] for example in batch]
            tgt_tokens = self.tokenizer.batch_encode_plus(
                targets,
                padding=True,
                return_tensors="pt",
                truncation=True,
            )
            target_ids = tgt_tokens["input_ids"]
            target_mask = tgt_tokens["attention_mask"].bool()
            target_ids = target_ids.masked_fill(~target_mask, -100)

            return {"input_ids": inp_tokens["input_ids"],
                    "attention_mask": inp_tokens["attention_mask"],
                    "labels": target_ids}

        elif self.task_type == "CAUSAL_LM":
            labels = inp_tokens["input_ids"].clone()
            if self.tokenizer.pad_token_id is not None:
                labels[labels == self.tokenizer.pad_token_id] = -100
            return {"input_ids": inp_tokens["input_ids"],
                    "attention_mask": inp_tokens["attention_mask"],
                    "labels": labels
                    }
        else:
            raise f"Unsupported task type for {self.task_type}"

    def tokenize_function(self, data: hfDataset, split: str=None,
                          perplexity_eval: bool=False):
        # if not perplexity_eval:
        #     inputs = data[self.text_column] + f" {self.tokenizer.eos_token}" if split != 'eval' or split != 'test' else data[self.text_column]
        # elif self.task_type == "CAUSAL_LM":
        #     inputs = data["perplexity"] + f" {self.tokenizer.eos_token}"
        # else:
        #     warnings.warn(f"Cannot do perplexity eval on {self.task_type}")
        #     pass

        if self.task_type == "CAUSAL_LM":
            if perplexity_eval:
                inputs = data["perplexity"] + f" {self.tokenizer.eos_token}"
            elif split == 'train':
                inputs = data[self.text_column] + f" {self.tokenizer.eos_token}"
            else:
                inputs = data[self.text_column]
        else:
            inputs = data[self.text_column]

        inp_tokens = self.tokenizer(
            inputs,
            # padding=False,
            return_special_tokens_mask=True,
            truncation="longest_first",
            max_length=self.model_max_length if split == "train" or perplexity_eval else self.context_length,
            # return_overflowing_tokens=True,
            # return_length=True,
        )

        if self.task_type == "SEQ_2_SEQ_LM":
            targets = data[self.target_column]
            tgt_tokens = self.tokenizer(
                targets,
                # padding=True,
                # return_tensors="pt",
                truncation="longest_first",
                return_special_tokens_mask=True,
                max_length=self.model_max_length if split == "train" else self.context_length
            )
            target_ids = tgt_tokens["input_ids"]
            target_mask = tgt_tokens["attention_mask"].bool()
            target_ids = target_ids.masked_fill(~target_mask, -100)

            return {"input_ids": inp_tokens["input_ids"],
                    "attention_mask": inp_tokens["attention_mask"],
                    "labels": target_ids}

        elif self.task_type == "CAUSAL_LM":
            return inp_tokens
        else:
            raise f"Unsupported task type for {self.task_type}"

    # Main data processing function that will concatenate all texts from our dataset and generate chunks of block_size.
    def group_texts(self, examples):
        # Concatenate all texts.
        concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        # We drop the small remainder, and if the total_length < block_size  we exclude this batch and return an empty dict.
        # We could add padding if the model supported it instead of this drop, you can customize this part to your needs.
        total_length = (total_length // self.block_size) * self.block_size
        # Split by chunks of max_len.
        result = {
            k: [t[i : i + self.block_size] for i in range(0, total_length, self.block_size)]
            for k, t in concatenated_examples.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    @staticmethod
    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2 ** 32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    def get_dataloader(self, dataset, shuffle_flag: bool = False, batch_size: int=1) -> DataLoader:
        """
        :param dataset: (Dataset): dataset from which to load the data.
        :param shuffle_flag: set to ``True`` to have the data reshuffled
                at every epoch (default: ``False``).
        :batch_size: The batch size of the dataset
        :return: a dataloder object

        Args:
            batch_size:

        """
        sampler = RandomSampler(data_source=dataset,
                                generator=self.generator) if shuffle_flag else SequentialSampler(dataset)

        if self.no_preprocess_data:
            collate_function = self.dynamic_collate
        elif self.task_type == "CAUSAL_LM":
            collate_function = DataCollatorForCompletionOnlyLM(self.response_template,
                                                               tokenizer=self.tokenizer,
                                                               mlm=False,
                                                               )
            # collate_function = DataCollatorForLanguageModeling(tokenizer=self.tokenizer, mlm=False)
        elif self.task_type == "SEQ_2_SEQ_LM":
            collate_function = DataCollatorForSeq2Seq(self.tokenizer)
        else:
            raise f"Unsupported task type for {self.task_type}"

        dist_print(f"Collate function {collate_function}")

        dataloader = DataLoader(dataset,
                                sampler=sampler,
                                collate_fn=collate_function,
                                batch_size=batch_size,
                                drop_last=False, # Keep this false for no model print evaluation mismatch
                                pin_memory=torch.cuda.is_available(),
                                worker_init_fn=self.seed_worker,
                                )

        return dataloader


if __name__ == "__main__":
    dataloader_args = {
        "model_name": "meta-llama/Llama-3.2-3B",
        "text_column": "prompt",
        "target_column": "target",
        "train_file": [
                       r"/Users/lochoang/Documents/Workspaces/capstone-project/llm/data/databricks-dolly-15k/databricks_dolly15k_translated.json",
                       r"/Users/lochoang/Documents/Workspaces/capstone-project/llm/data/databricks-dolly-15k/databricks_dolly15k.json",
                        ],
        "val_file": [
            r"/Users/lochoang/Documents/Workspaces/capstone-project/llm/data/databricks-dolly-15k/databricks_dolly15k_translated.json",
            # r"src/data/features/final_storge_converted/Open-Orca_OpenOrca/OpenOrcaFormated.json"
        ],
        "train_batch_size": 8,
        "perplexity_eval_batch_size": 6,
        "generative_eval_batch_size": 8,
        "seed": 42,
        "max_train_samples": 450,
        "max_eval_samples": 200,
        "config_type": AdvanceInstructSample,
        "task_type": "CAUSAL_LM",
        "no_preprocess_data": False,
        "do_group_texts": False,
        "do_perplexity_eval": True,
        "do_generative_eval": True
    }

    # qa_dataset = AdvanceQa(json_file_paths=[
    #                                         r"src/data/features/final_storge_converted/Open-Orca_OpenOrca/OpenOrca_translatedFormated.json",
    #                                         r"src/data/features/final_storge_converted/Open-Orca_OpenOrca/OpenOrcaFormated.json"],
    #                        num_examples=5000,
    #                        config_type=AdvanceInstructSample,
    #                        get_example=False,
    #                        task_type="CAUSAL_LM")
    # print(qa_dataset[idx])
    # print(qa_dataset[idx].get_dict)
    # print(qa_dataset[idx].get_dict_str)
    # for i in range(0, 20):
    #     idx = random.randint(0, 5000)
    #     # prompt = qa_dataset[idx].get_example(is_training=True, task_type="CAUSAL_LM")
    #     # if len(prompt['prompt']) < 768 and len(prompt['prompt']) > 512:
    #     #     print(prompt)
    #     prompt = qa_dataset[idx]
    #     print(prompt)

    qa_dataloader = QADataloader(**dataloader_args)
    qa_dataloader_instance = qa_dataloader.__call__()
    print(qa_dataloader.dataset)

    # for idx, data in enumerate(iter(qa_dataloader_instance['eval']['generative_eval'])):
    #     print("\n"+qa_dataloader.tokenizer.decode(data['input_ids'][0], skip_special_tokens=True))
    #     labels = data['labels'].cpu().numpy()
    #     labels = np.where(labels != -100, labels, qa_dataloader.tokenizer.pad_token_id)
    #     print("\n"+qa_dataloader.tokenizer.decode(labels[0], skip_special_tokens=True))
    #     if idx == 10: break

    # for idx, data in enumerate(iter(qa_dataloader_instance['eval']['perplexity_eval'])):
    #     print("\n"+qa_dataloader.tokenizer.decode(data['input_ids'][0], skip_special_tokens=True))
    #     labels = data['labels'].cpu().numpy()
    #     labels = np.where(labels != -100, labels, qa_dataloader.tokenizer.pad_token_id)
    #     print("\n"+qa_dataloader.tokenizer.decode(labels[0], skip_special_tokens=True))
    #     if idx == 20: break


 meta-llama/Llama-3.2-3B's tokenizer does not have pad_token token, setting it to [PAD]

 meta-llama/Llama-3.2-3B's tokenizer does not have unk_token token, setting it to <unk>


Loading train datasets..........


Loading train data:   0%|          | 0/2 [00:00<?, ?it/s]

Loading data from databricks_dolly15k_translated.json for split train:   0%|          | 0/225 [00:00<?, ?it/s]

Loading 225 examples with percentage of 50 from databricks_dolly15k_translated.json...

Finished loading from databricks_dolly15k_translated.json with total loaded 226 examples

Total data skipped: 0



Loading data from databricks_dolly15k.json for split train:   0%|          | 0/225 [00:00<?, ?it/s]

Loading 225 examples with percentage of 50 from databricks_dolly15k.json...

Finished loading from databricks_dolly15k.json with total loaded 452 examples

Total data skipped: 3

Sample 248 of the training set: {'prompt': ' \n\n\n ####### Instruction:\nWhat happens when the sun goes down?\n\n %%%%%%% Response:\nWhen the sun sets, the evening starts.\n'}.
Sample 42 of the training set: {'prompt': ' \n\n\n ####### Instruction:\nMô tả một giấc mơ lặp đi lặp lại mà bạn đã có.\n\n %%%%%%% Response:\nTrong suốt cuộc đời mình, tôi luôn có một giấc mơ lặp đi lặp lại là tôi bị truy đuổi. Đôi khi là bởi thây ma, đôi khi là bởi cảnh sát, đôi khi là bởi những người cộng sản (ông tôi thường bắt tôi xem đi xem lại Red Dawn khi còn nhỏ). Thông thường nó sẽ bắt đầu ở ngoài trời, và tôi sẽ trốn sau những tán cây, chạy đua qua những cây cầu ọp ẹp và nhảy qua những con mương. Cuối cùng, nó luôn kết thúc ở một ngôi nhà cổ rộng lớn với nhiều cầu thang và tủ quần áo. Thường thì hành lang sẽ thu hẹp lại thàn

Loading eval data:   0%|          | 0/1 [00:00<?, ?it/s]

Loading data from databricks_dolly15k_translated.json for split eval:   0%|          | 0/200 [00:00<?, ?it/s]

Loading 200 examples with percentage of 100 from databricks_dolly15k_translated.json...

Finished loading from databricks_dolly15k_translated.json with total loaded 201 examples

Total data skipped: 0

Sample 158 of the training set: {'prompt': ' \nYou are functioning as an AI assistant. Your objective is to produce a relevant answer.\n\n ####### Instruction:\nLựa chọn hẹn hò đầu tiên ở Boston là gì?\n\n %%%%%%% Response:\n', 'target': 'Có nhiều lựa chọn cho buổi hẹn hò đầu tiên ở Boston. Bạn có thể đi dạo thoải mái ở Boston Common và thưởng thức cà phê/trà ở đâu đó gần đó. Bạn có thể đến Thủy cung Boston và đi bộ dọc theo mặt nước sau đó. Bạn có thể đến Seaport và dành thời gian chơi gôn mini trong nhà. Bạn cũng có thể đi lang thang quanh chợ Quincy và ăn một miếng rồi ghé thăm nhà máy bia Sam Adams. Cuối cùng, bạn có thể đi dạo dọc theo lối đi dạo và thưởng thức đồ uống cũng như nhà máy bia Night Shift nếu thời tiết đẹp.\n', 'perplexity': ' \nYou are functioning as an AI assistant. Y

In [ ]:
import json
import os.path
import sys
from typing import Union, List
sys.path.insert(0, r'./')


def reformat_data(data_paths: List[str], added_string: str="Formated"):
    "Format json data to supported data type for pyarrow"

    for file in data_paths:
        assert os.path.isfile(file), f"Please provide the correct path, No path exist for {file}"
        with open(file, "r", encoding="utf-8") as f:
            json_data = [json.loads(line) for line in f]
            file_name = os.path.basename(file).split(".")[0]
            formated_file = file.replace(file_name+".", file_name+added_string+".")
            with open(formated_file, "w", encoding="utf-8") as f:
                for item in json_data:
                    json.dump(item, f, ensure_ascii=False)
                    f.write("\n")
        print(f"Finished converted {file}")


if __name__=="__main__":
    reformat_data([r"/Users/lochoang/Documents/Workspaces/capstone-project/llm/data/databricks_dolly15k.json",
                   r"/Users/lochoang/Documents/Workspaces/capstone-project/llm/data/databricks_dolly15k_translated.json"])

Finished converted /Users/lochoang/Documents/Workspaces/capstone-project/llm/data/databricks_dolly15k.json
Finished converted /Users/lochoang/Documents/Workspaces/capstone-project/llm/data/databricks_dolly15k_translated.json
